## Building a Custom Objects Detection with YOLO

Using the `pre-trained` YOLO model to detect objects in our Dhaka traffic video directly will limit us to `detecting only the classes that the model already knows about`. In order to detect all the various classes of vehicle that we want, **we'll need to train a classifier ourselves**.

In this notebook we will use `transfer learning` to fine-tune a pre-trained `YOLO` model on the object classes that we care about. . This requires us to transform the XML-formatted bounding boxes into a new format that the YOLO model requires. With these transformed labels, we can run a training loop on the YOLO model. After training, we'll be able to detect our object classes in the video frames.

Note that the objective is to:
- convert bounding boxes to a new representation
- Assemble files into the desired directory structure
- Cleanly handle malformed data
- Fine-tune a model to detect new classes
- Detect these new classes in images

YAML: is a text format that YOLO accepts.

## Packages Import

First we need to install `ultralytics`
 using `!pip install ultralytics` if its missing!

In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 3.9 MB/s eta 0:00:00


## Import all packages and dependencies

In [2]:
import pathlib
from pathlib import Path
import random
import shutil
import sys
import json
import csv

import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import torch
import ultralytics
import yaml
from IPython import display
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from tqdm.auto import tqdm
from ultralytics import YOLO
from collections import Counter
import xml.etree.ElementTree as ET
from collections import Counter
from IPython.display import display as ipy_display
import torch
from ultralytics import YOLO
from zipfile import ZipFile, ZIP_DEFLATED

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In case we want to reproduce this notebook in the future, we'll record the version information.

In [3]:
print("Platform:", sys.platform)
print("Python version:", sys.version)
print("---")
print("matplotlib version:", plt.matplotlib.__version__)
print("pandas version:", pd.__version__)
print("PIL version:", Image.__version__)
print("PyYAML version:", yaml.__version__)
print("torch version:", torch.__version__)
print("ultralytics version:", ultralytics.__version__)

Platform: linux
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
---
matplotlib version: 3.10.0
pandas version: 2.3.3
PIL version: 12.3.0
PyYAML version: 6.0.3
torch version: 2.11.0+cu128
ultralytics version: 8.4.183


We already have a GPU available in this kaggle notebook. Since this will require a GPU to run in any reasonable amount of time. In case you are not sure use the code in the notebook below to confirm and switch device to `cuda`.

In [4]:
# if torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

# print(f"Using {device} device.")

## Data Loading

The [Dhaka AI](https://www.kaggle.com/datasets/rifat963/dhakaai-dhaka-based-traffic-detection-dataset) data set will provide the training images and labels we need to train a model. Rather than starting from scratch, we'll conduct transfer learning with a pre-trained YOLO model. This will take a while to train, but it'll be significantly faster than training from scratch!

Since I'll be using kaggle noteboook to train the YOLO model, I have already added the data into the notebook. `Let's load the data`.

`Important`: Kaggle's `/kaggle/input/` directory is read-only. We can inspect the dataset there, but if we need to reorganize or convert its files, we must create a working copy under `/kaggle/working/`.

### List all folders under /kaggle/input

In [5]:
INPUT_DIR = Path("/kaggle/input")

print("Does /kaggle/input exist?", INPUT_DIR.exists())

if INPUT_DIR.exists():
    for item in sorted(INPUT_DIR.iterdir()):
        print(item.name, "->", item)
        if item.is_dir():
            for child in sorted(item.iterdir()):
                print("    ", child.name, "->", child)

Does /kaggle/input exist? True
datasets -> /kaggle/input/datasets
     rifat963 -> /kaggle/input/datasets/rifat963


### Let's inspect the dataset structure

In [6]:
ROOT = Path("/kaggle/input/datasets/rifat963")

print("Top-level folders:")
for p in ROOT.iterdir():
    if p.is_dir():
        print(f"📁 {p.name}/")

print("\nFolders up to 3 levels deep:")
for p in ROOT.rglob("*"):
    if p.is_dir() and len(p.relative_to(ROOT).parts) <= 3:
        depth = len(p.relative_to(ROOT).parts)
        print("    " * (depth - 1) + f"📁 {p.name}/")

Top-level folders:
📁 dhakaai-dhaka-based-traffic-detection-dataset/

Folders up to 3 levels deep:
📁 dhakaai-dhaka-based-traffic-detection-dataset/
    📁 test/
    📁 test round 2/
    📁 train/
        📁 test/
        📁 test round 2/
        📁 Final Train Dataset/


In [7]:
# Inspect the dataset structure inside `dhakaai-dhaka-based-traffic-detection-dataset/`
DATA_DIR = Path(
    "/kaggle/input/datasets/rifat963/"
    "dhakaai-dhaka-based-traffic-detection-dataset"
)

# Display folders and only the first 5 files in each folder
for folder in sorted(DATA_DIR.rglob("*")):
    if folder.is_dir():
        files = [p.name for p in folder.iterdir() if p.is_file()]
        subfolders = [p.name for p in folder.iterdir() if p.is_dir()]

        if files or subfolders:
            print(f"\n📁 {folder.relative_to(DATA_DIR)}")
            print(f"   Subfolders: {subfolders[:10]}")
            print(f"   Number of files: {len(files)}")
            if files:
                print(f"   Sample files: {files[:5]}")


📁 test
   Subfolders: ['test']
   Number of files: 0

📁 test/test
   Subfolders: []
   Number of files: 500
   Sample files: ['Asraf_50_jpg.rf.7026694f0b9f37a6790982295c7e8663.jpg', 'Asraf_51_jpg.rf.0e3516baf7509bc2c4a4aa8deea494c2.jpg', 'Asraf_52_jpg.rf.867869f276e6db3a09a84b99643df316.jpg', 'Sabiha_(01)_jpg.rf.f935c7fc51a14c64e34c17a17c41cb1f.jpg', 'Sabiha_(10)_jpg.rf.d163e2a53d5ed22112e9ead62807acf4.jpg']

📁 test round 2
   Subfolders: ['test round 2']
   Number of files: 0

📁 test round 2/test round 2
   Subfolders: []
   Number of files: 450
   Sample files: ['Asraf_100_jpg.rf.472d5bd950587612520051eebd827328.jpg', 'Asraf_101_jpg.rf.58690ebe018b849e3d3f6ef8785ca148.jpg', 'Asraf_102_jpg.rf.40d695ed6f8db102530afcb11db466fa.jpg', 'Asraf_103_jpg.rf.19874eeb1f0a60c783cf34375a7c724b.jpg', 'Asraf_104_jpg.rf.a5288ec7a648c79f1806a0df2cddeb55.jpg']

📁 train
   Subfolders: ['Final Train Dataset']
   Number of files: 0

📁 train/Final Train Dataset
   Subfolders: []
   Number of files: 6006
 

The output reveals an important detail: this appears to be an object-detection dataset with Pascal VOC-style XML annotations. The train/Final Train Dataset folder contains 6,006 files, and our sample includes Numan_(470).xml. The two test folders contain 500 and 450 images respectively. Before training a model such as YOLO, we should verify the image-to-annotation pairing and inspect the XML structure. We should also avoid assuming that every file in the training folder is an image.

### Count images and annotations

In [8]:
DATA_DIR = Path(
    "/kaggle/input/datasets/rifat963/"
    "dhakaai-dhaka-based-traffic-detection-dataset"
)

TRAIN_DIR = DATA_DIR / "train" / "Final Train Dataset"

# Count file types
files = [p for p in TRAIN_DIR.iterdir() if p.is_file()]
extensions = Counter(p.suffix.lower() for p in files)

print("Total files:", len(files))
print("\nFile types:")
for ext, count in extensions.most_common():
    print(f"{ext or '[no extension]'}: {count}")

# Find images and XML annotations
image_exts = {".jpg", ".jpeg", ".png", ".bmp"}
images = [p for p in files if p.suffix.lower() in image_exts]
xml_files = [p for p in files if p.suffix.lower() == ".xml"]

image_stems = {p.stem for p in images}
xml_stems = {p.stem for p in xml_files}

print("\nImage count:", len(images))
print("XML annotation count:", len(xml_files))
print("Images with matching XML:", len(image_stems & xml_stems))
print("Images without XML:", len(image_stems - xml_stems))
print("XML without matching image:", len(xml_stems - image_stems))

# Inspect one XML annotation
if xml_files:
    xml_path = xml_files[0]
    print(f"\nSample annotation: {xml_path.name}")

    try:
        root = ET.parse(xml_path).getroot()

        print("Image filename:", root.findtext("filename"))
        print("Image width:", root.findtext("size/width"))
        print("Image height:", root.findtext("size/height"))

        for obj in root.findall("object"):
            print("Class:", obj.findtext("name"))
            box = obj.find("bndbox")
            if box is not None:
                print("Bounding box:", {
                    key: box.findtext(key)
                    for key in ("xmin", "ymin", "xmax", "ymax")
                })
    except ET.ParseError as e:
        print("Could not parse XML:", e)


Total files: 6006

File types:
.xml: 3003
.jpg: 2987
.png: 14
.jpeg: 2

Image count: 3003
XML annotation count: 3003
Images with matching XML: 3003
Images without XML: 0
XML without matching image: 0

Sample annotation: 01.xml
Image filename: 02_Motijheel_280714_0005.jpg
Image width: 1200
Image height: 800
Class: bus
Bounding box: {'xmin': '833', 'ymin': '390', 'xmax': '1087', 'ymax': '800'}
Class: bus
Bounding box: {'xmin': '901', 'ymin': '284', 'xmax': '1018', 'ymax': '395'}
Class: bus
Bounding box: {'xmin': '909', 'ymin': '241', 'xmax': '1010', 'ymax': '287'}
Class: rickshaw
Bounding box: {'xmin': '761', 'ymin': '413', 'xmax': '832', 'ymax': '540'}
Class: rickshaw
Bounding box: {'xmin': '777', 'ymin': '364', 'xmax': '828', 'ymax': '409'}
Class: rickshaw
Bounding box: {'xmin': '120', 'ymin': '351', 'xmax': '177', 'ymax': '423'}
Class: rickshaw
Bounding box: {'xmin': '178', 'ymin': '340', 'xmax': '245', 'ymax': '419'}
Class: rickshaw
Bounding box: {'xmin': '551', 'ymin': '229', 'xmax'

Excellent, we can see the audit confirms that the training directory contains 3,003 images and 3,003 matching XML annotations, with no missing image annotation pairs. We can proceed with preparing the dataset for YOLO object detection. Our annotations contain multiple objects per image, with classes such as `suv`, `three wheelers (CNG)`, and `car`. The example also confirms that the XML files contain image dimensions and bounding-box coordinates. 

### Let's audit the classes and bounding boxes

Before converting annotations, we should identify every class, the number of objects per class, and any invalid bounding boxes. This helps prevent incorrect class mappings or training failures.

In [9]:
class_counts = Counter()
invalid_boxes = []
empty_annotations = []
malformed_xml = []

for xml_path in TRAIN_DIR.glob("*.xml"):
    try:
        root = ET.parse(xml_path).getroot()
        width = int(root.findtext("size/width"))
        height = int(root.findtext("size/height"))
        objects = root.findall("object")

        if not objects:
            empty_annotations.append(xml_path.name)

        for obj in objects:
            cls = (obj.findtext("name") or "").strip()
            class_counts[cls] += 1

            box = obj.find("bndbox")
            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))
            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))

            if (
                not cls
                or xmin < 0 or ymin < 0
                or xmax > width or ymax > height
                or xmin >= xmax or ymin >= ymax
            ):
                invalid_boxes.append(
                    (xml_path.name, cls, xmin, ymin, xmax, ymax)
                )

    except Exception as e:
        malformed_xml.append((xml_path.name, str(e)))

print("Number of classes:", len(class_counts))
print("\nCLASS MAPPING CANDIDATES")
for class_id, (name, count) in enumerate(
    sorted(class_counts.items())
):
    print(f"{class_id}: {name} — {count} objects")

print("\nTotal annotated objects:", sum(class_counts.values()))
print("Invalid bounding boxes:", len(invalid_boxes))
print("Empty annotations:", len(empty_annotations))
print("Malformed XML files:", len(malformed_xml))

if invalid_boxes:
    print("\nFirst 5 invalid boxes:")
    print(*invalid_boxes[:5], sep="\n")

if malformed_xml:
    print("\nFirst 5 malformed files:")
    print(*malformed_xml[:5], sep="\n")


Number of classes: 21

CLASS MAPPING CANDIDATES
0: ambulance — 70 objects
1: army vehicle — 43 objects
2: auto rickshaw — 372 objects
3: bicycle — 459 objects
4: bus — 3340 objects
5: car — 5476 objects
6: garbagevan — 3 objects
7: human hauler — 169 objects
8: minibus — 95 objects
9: minivan — 935 objects
10: motorbike — 2284 objects
11: pickup — 1225 objects
12: policecar — 32 objects
13: rickshaw — 3549 objects
14: scooter — 38 objects
15: suv — 860 objects
16: taxi — 60 objects
17: three wheelers (CNG) — 2990 objects
18: truck — 1492 objects
19: van — 756 objects
20: wheelbarrow — 120 objects

Total annotated objects: 24368
Invalid bounding boxes: 24
Empty annotations: 0
Malformed XML files: 1

First 5 invalid boxes:
('Pias (359).xml', 'minivan', 1.0, 241.0, 465.0, 515.0)
('Pias (359).xml', 'suv', 508.0, 255.0, 747.0, 460.0)
('Pias (359).xml', 'rickshaw', 1.0, 268.0, 393.0, 614.0)
('Pias (359).xml', 'rickshaw', 679.0, 241.0, 727.0, 304.0)
('Pias (359).xml', 'rickshaw', 751.0, 242.0

The audit has identified the main issues to address before training our YOLO detector.

- 21 object classes and 24,368 annotated objects.
- 3,003 image/XML pairs in the training directory.
- 24 invalid bounding boxes reported by the audit, likely including boxes extending beyond image boundaries.
- 1 malformed XML file: 231.xml.
- No empty annotations were found.
  
One important correction: the 24 invalid boxes need inspection before deciding how to handle them. Some can be safely clipped to image boundaries; others may need to be excluded if the annotation is unusable. We should also inspect 231.xml to determine whether it is genuinely corrupt.

We'll preserve the original dataset and create a separate YOLO-ready copy, so preprocessing does not modify your source files.

---

We'll build a reproducible YOLO dataset pipeline for our Dhaka traffic detection project using 1,500 selected images, an 80/20 split, and all 21 classes as the coverage objective.
The pipeline will preserve the original dataset and both test folders, use class-aware image selection, convert Pascal VOC XML annotations into YOLO labels, and write audit logs for malformed XML, invalid boxes, excluded images, and class distributions.

One important limitation: class-aware sampling can prioritize all 21 classes, but it cannot guarantee that every class appears in both subsets-especially `garbagevan`, which has only three annotated objects across the original dataset. We'll measure actual coverage rather than assume it.

In [10]:
SEED = 42
SAMPLE_SIZE = 1500
TRAIN_RATIO = 0.80

DATA_DIR = Path(
    "/kaggle/input/datasets/rifat963/"
    "dhakaai-dhaka-based-traffic-detection-dataset"
)

TRAIN_SOURCE = DATA_DIR / "train" / "Final Train Dataset"

OUTPUT_DIR = Path("/kaggle/working/dhakaai_yolo_1500_seed42")

if not TRAIN_SOURCE.exists():
    raise FileNotFoundError(
        f"Training folder not found: {TRAIN_SOURCE}\n"
        "Correct DATA_DIR to match your Kaggle input path."
    )

if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"{OUTPUT_DIR} already exists. Rename it or remove it "
        "after verifying it contains only generated files."
    )

random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

print("Dataset:", DATA_DIR)
print("Training source:", TRAIN_SOURCE)
print("Output:", OUTPUT_DIR)
print("Seed:", SEED)

Dataset: /kaggle/input/datasets/rifat963/dhakaai-dhaka-based-traffic-detection-dataset
Training source: /kaggle/input/datasets/rifat963/dhakaai-dhaka-based-traffic-detection-dataset/train/Final Train Dataset
Output: /kaggle/working/dhakaai_yolo_1500_seed42
Seed: 42


In [11]:
#Making sure the path is correct
assert TRAIN_SOURCE.is_dir(), f"Training folder not found: {TRAIN_SOURCE}"
assert not OUTPUT_DIR.exists(), (
    f"Output already exists: {OUTPUT_DIR}. Choose a new output directory."
)

### Parse annotations and audit image files

This cell reads the XML files, verifies image dimensions, repairs boxes that can be clipped safely, and records malformed annotations. It does not modify the source files.

In [12]:
# Parse annotations and validate bounding boxes

image_files = sorted(
    p for p in TRAIN_SOURCE.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)
xml_files = sorted(TRAIN_SOURCE.glob("*.xml"))

images_by_stem = {}
for path in image_files:
    images_by_stem.setdefault(path.stem, []).append(path)

records = []
image_audit = []
box_audit = []
class_names_set = set()

for xml_path in xml_files:
    try:
        root = ET.parse(xml_path).getroot()
    except Exception as exc:
        image_audit.append({
            "xml": xml_path.name,
            "image": "",
            "status": "malformed_xml",
            "details": str(exc),
        })
        continue

    candidates = images_by_stem.get(xml_path.stem, [])
    if len(candidates) != 1:
        image_audit.append({
            "xml": xml_path.name,
            "image": "",
            "status": "missing_or_ambiguous_image",
            "details": f"Matching images: {len(candidates)}",
        })
        continue

    image_path = candidates[0]

    try:
        with Image.open(image_path) as im:
            actual_w, actual_h = im.size
            im.verify()
    except Exception as exc:
        image_audit.append({
            "xml": xml_path.name,
            "image": image_path.name,
            "status": "unreadable_image",
            "details": str(exc),
        })
        continue

    try:
        xml_w = int(root.findtext("size/width"))
        xml_h = int(root.findtext("size/height"))
        if xml_w <= 0 or xml_h <= 0:
            raise ValueError("Invalid XML image dimensions")
    except Exception as exc:
        image_audit.append({
            "xml": xml_path.name,
            "image": image_path.name,
            "status": "invalid_xml_dimensions",
            "details": str(exc),
        })
        continue

    dimension_mismatch = (xml_w != actual_w or xml_h != actual_h)
    sx = actual_w / xml_w
    sy = actual_h / xml_h
    yolo_objects = []

    for obj_idx, obj in enumerate(root.findall("object")):
        cls = (obj.findtext("name") or "").strip()
        box = obj.find("bndbox")

        if not cls or box is None:
            box_audit.append({
                "xml": xml_path.name,
                "object_index": obj_idx,
                "class": cls,
                "status": "missing_class_or_box",
            })
            continue

        try:
            # Scale XML coordinates if declared dimensions differ
            # from the actual image dimensions.
            xmin = float(box.findtext("xmin")) * sx
            ymin = float(box.findtext("ymin")) * sy
            xmax = float(box.findtext("xmax")) * sx
            ymax = float(box.findtext("ymax")) * sy
        except (TypeError, ValueError):
            box_audit.append({
                "xml": xml_path.name,
                "object_index": obj_idx,
                "class": cls,
                "status": "invalid_coordinates",
            })
            continue

        original = (xmin, ymin, xmax, ymax)

        # Clip only if the box intersects the image.
        xmin = max(0.0, min(xmin, actual_w))
        ymin = max(0.0, min(ymin, actual_h))
        xmax = max(0.0, min(xmax, actual_w))
        ymax = max(0.0, min(ymax, actual_h))

        if xmin >= xmax or ymin >= ymax:
            box_audit.append({
                "xml": xml_path.name,
                "object_index": obj_idx,
                "class": cls,
                "status": "discarded_invalid_after_clipping",
                "original_box": str(original),
            })
            continue

        if original != (xmin, ymin, xmax, ymax) or dimension_mismatch:
            box_status = (
                "clipped_and_scaled"
                if dimension_mismatch and original != (xmin, ymin, xmax, ymax)
                else "scaled_dimensions"
                if dimension_mismatch
                else "clipped"
            )
        else:
            box_status = "valid"

        # Standard YOLO normalized center/width/height format.
        xc = ((xmin + xmax) / 2) / actual_w
        yc = ((ymin + ymax) / 2) / actual_h
        bw = (xmax - xmin) / actual_w
        bh = (ymax - ymin) / actual_h

        yolo_objects.append({
            "class": cls,
            "box": (xc, yc, bw, bh),
        })
        class_names_set.add(cls)

        box_audit.append({
            "xml": xml_path.name,
            "object_index": obj_idx,
            "class": cls,
            "status": box_status,
            "original_box": str(original),
            "final_box": str((xmin, ymin, xmax, ymax)),
        })

    if not yolo_objects:
        image_audit.append({
            "xml": xml_path.name,
            "image": image_path.name,
            "status": "excluded_no_valid_objects",
            "details": "",
        })
        continue

    records.append({
        "image_path": image_path,
        "xml_path": xml_path,
        "width": actual_w,
        "height": actual_h,
        "objects": yolo_objects,
        "dimension_mismatch": dimension_mismatch,
    })

    image_audit.append({
        "xml": xml_path.name,
        "image": image_path.name,
        "status": "usable",
        "details": (
            f"{len(yolo_objects)} valid objects; "
            f"dimension_mismatch={dimension_mismatch}"
        ),
    })

if not records:
    raise RuntimeError("No usable annotated images found.")

# Fixed alphabetical class mapping.
class_names = sorted(class_names_set)
class_to_id = {name: idx for idx, name in enumerate(class_names)}

if len(class_names) != 21:
    print(
        f"WARNING: Found {len(class_names)} classes in usable "
        "annotations, rather than the expected 21."
    )

print("Source images:", len(image_files))
print("XML files:", len(xml_files))
print("Usable annotated images:", len(records))
print("Usable classes:", len(class_names))
print("Classes:", class_to_id)
print("Malformed XML:", sum(
    row["status"] == "malformed_xml" for row in image_audit
))
print("Dimension mismatches:", sum(
    r["dimension_mismatch"] for r in records
))
print("Bounding-box audit entries:", len(box_audit))

Source images: 3003
XML files: 3003
Usable annotated images: 3000
Usable classes: 21
Classes: {'ambulance': 0, 'army vehicle': 1, 'auto rickshaw': 2, 'bicycle': 3, 'bus': 4, 'car': 5, 'garbagevan': 6, 'human hauler': 7, 'minibus': 8, 'minivan': 9, 'motorbike': 10, 'pickup': 11, 'policecar': 12, 'rickshaw': 13, 'scooter': 14, 'suv': 15, 'taxi': 16, 'three wheelers (CNG)': 17, 'truck': 18, 'van': 19, 'wheelbarrow': 20}
Malformed XML: 1
Dimension mismatches: 3
Bounding-box audit entries: 24344


### Select 1,500 images with class-aware sampling

The sampler prioritizes images containing classes that appear in fewer images. It then reduces the priority of classes already well represented in the selected subset. This is more useful than sampling purely at random for a dataset with rare classes.

In [13]:
# Reproducible class-aware selection and 80/20 split

if len(records) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(records)} usable images; cannot select {SAMPLE_SIZE}."
    )

rng = np.random.default_rng(SEED)
n = len(records)
class_index = {name: i for i, name in enumerate(class_names)}

# Image-level multi-label matrix: 1 if the image contains a class.
presence = np.zeros((n, len(class_names)), dtype=np.int32)

for i, record in enumerate(records):
    for obj in record["objects"]:
        presence[i, class_index[obj["class"]]] = 1

# Greedy sampling with higher initial weight for rare classes.
available = np.ones(n, dtype=bool)
selected_indices = []
selected_class_images = np.zeros(len(class_names), dtype=np.float64)
total_class_images = presence.sum(axis=0).astype(np.float64)

for _ in range(SAMPLE_SIZE):
    candidates = np.flatnonzero(available)

    # Rarer classes get more weight; priority decreases as they
    # become represented in the selected set.
    weights = 1.0 / np.sqrt(
        np.maximum(total_class_images, 1)
        * (selected_class_images + 1)
    )
    scores = presence[candidates] @ weights

    # Seeded tie-breaking for reproducibility.
    scores += rng.random(len(candidates)) * 1e-10
    chosen = candidates[int(np.argmax(scores))]

    selected_indices.append(int(chosen))
    available[chosen] = False
    selected_class_images += presence[chosen]

selected_records = [records[i] for i in selected_indices]
selected_presence = presence[selected_indices]
selected_n = len(selected_records)

# Select exactly 20% of selected images for validation.
val_size = round(selected_n * (1 - TRAIN_RATIO))
train_size = selected_n - val_size

val_class_total = selected_presence.sum(axis=0).astype(int)

# For a class occurring in >=2 selected images, aim for at least
# one validation image. Classes appearing in only one image stay
# in training to avoid removing their only training example.
val_targets = np.zeros(len(class_names), dtype=int)
for j, count in enumerate(val_class_total):
    if count >= 2:
        val_targets[j] = max(1, round(count * (1 - TRAIN_RATIO)))

remaining = np.ones(selected_n, dtype=bool)
val_indices = []
val_class_selected = np.zeros(len(class_names), dtype=int)

while len(val_indices) < val_size:
    candidates = np.flatnonzero(remaining)
    deficits = np.maximum(val_targets - val_class_selected, 0)

    if deficits.sum() == 0:
        # All attainable class targets met; fill remaining slots
        # reproducibly without replacement.
        needed = val_size - len(val_indices)
        chosen = rng.choice(candidates, size=needed, replace=False)
        val_indices.extend(int(i) for i in chosen)
        break

    # Prioritize outstanding deficits, accounting for how many
    # remaining images could satisfy each class.
    remaining_class_images = selected_presence[candidates].sum(axis=0)
    weights = deficits / np.maximum(remaining_class_images, 1)
    scores = selected_presence[candidates] @ weights
    scores += rng.random(len(candidates)) * 1e-10

    chosen = int(candidates[int(np.argmax(scores))])
    val_indices.append(chosen)
    remaining[chosen] = False
    val_class_selected += selected_presence[chosen]

val_set = set(val_indices)
train_records = [
    record for i, record in enumerate(selected_records)
    if i not in val_set
]
val_records = [
    selected_records[i] for i in val_indices
]

assert len(train_records) == train_size
assert len(val_records) == val_size
assert len(set(r["image_path"] for r in train_records)) == train_size
assert len(set(r["image_path"] for r in val_records)) == val_size

print(f"Selected: {selected_n}")
print(f"Training: {len(train_records)}")
print(f"Validation: {len(val_records)}")

selected_classes = {
    obj["class"]
    for record in selected_records
    for obj in record["objects"]
}
missing_classes = sorted(set(class_names) - selected_classes)
print("Classes represented in selected subset:", len(selected_classes))
print("Classes absent from selected subset:", missing_classes)


Selected: 1500
Training: 1200
Validation: 300
Classes represented in selected subset: 21
Classes absent from selected subset: []


### Create YOLO images, labels, data.yaml, and audit logs

The cell below copies only the selected images into the new dataset. It writes one .txt file per image and preserves our original test and test round 2 directories untouched.

In [14]:
# Build YOLO dataset and save audit logs
# Create output directories
for split in ("train", "val"):
    (OUTPUT_DIR / "images" / split).mkdir(
        parents=True, exist_ok=True
    )
    (OUTPUT_DIR / "labels" / split).mkdir(
        parents=True, exist_ok=True
    )

def write_split(split_name, split_records):
    image_dir = OUTPUT_DIR / "images" / split_name
    label_dir = OUTPUT_DIR / "labels" / split_name

    for record in tqdm(
        split_records,
        desc=f"Preparing {split_name}",
        unit="image"
    ):
        image_path = record["image_path"]
        label_path = label_dir / f"{image_path.stem}.txt"
        destination = image_dir / image_path.name

        # Avoid recopying an image if the cell is rerun.
        if not destination.exists():
            shutil.copy2(image_path, destination)

        # Write YOLO labels.
        with open(label_path, "w", encoding="utf-8") as f:
            for obj in record["objects"]:
                class_id = class_to_id[obj["class"]]
                xc, yc, bw, bh = obj["box"]

                f.write(
                    f"{class_id} {xc:.6f} {yc:.6f} "
                    f"{bw:.6f} {bh:.6f}\n"
                )

write_split("train", train_records)
write_split("val", val_records)

print("Image copying and label generation complete.")

# Write YOLO configuration
yaml_text = (
    f"path: {json.dumps(str(OUTPUT_DIR))}\n"
    "train: images/train\n"
    "val: images/val\n"
    f"nc: {len(class_names)}\n"
    f"names: {json.dumps(class_names, ensure_ascii=False)}\n"
)

(OUTPUT_DIR / "data.yaml").write_text(
    yaml_text, encoding="utf-8"
)

with open(
    OUTPUT_DIR / "class_mapping.json", "w", encoding="utf-8"
) as f:
    json.dump(class_to_id, f, indent=2, ensure_ascii=False)

# Save audit reports
print("Saving audit reports...")

pd.DataFrame(image_audit).to_csv(
    OUTPUT_DIR / "image_audit.csv", index=False
)

pd.DataFrame(box_audit).to_csv(
    OUTPUT_DIR / "bounding_box_audit.csv", index=False
)

selection_rows = []

for split_name, split_records in [
    ("train", train_records),
    ("val", val_records),
]:
    for record in split_records:
        selection_rows.append({
            "image": record["image_path"].name,
            "xml": record["xml_path"].name,
            "split": split_name,
            "width": record["width"],
            "height": record["height"],
            "objects": len(record["objects"]),
            "classes": "; ".join(sorted({
                obj["class"] for obj in record["objects"]
            })),
        })

pd.DataFrame(selection_rows).to_csv(
    OUTPUT_DIR / "selected_images.csv", index=False
)

# Efficiently calculate class distribution in one pass per split.
class_summary = []

for class_name in tqdm(class_names, desc="Class distribution"):
    row = {"class": class_name}

    for split_name, split_records in [
        ("train", train_records),
        ("val", val_records),
    ]:
        object_count = 0
        image_count = 0

        for record in split_records:
            count = sum(
                obj["class"] == class_name
                for obj in record["objects"]
            )
            object_count += count
            image_count += int(count > 0)

        row[f"{split_name}_objects"] = object_count
        row[f"{split_name}_images"] = image_count

    class_summary.append(row)

pd.DataFrame(class_summary).to_csv(
    OUTPUT_DIR / "class_distribution.csv", index=False
)

summary = {
    "seed": SEED,
    "requested_sample_size": SAMPLE_SIZE,
    "selected_images": len(selected_records),
    "train_images": len(train_records),
    "val_images": len(val_records),
    "number_of_classes": len(class_names),
    "class_names": class_names,
    "selected_objects": sum(
        len(r["objects"]) for r in selected_records
    ),
    "train_objects": sum(
        len(r["objects"]) for r in train_records
    ),
    "val_objects": sum(
        len(r["objects"]) for r in val_records
    ),
    "classes_missing_from_selection": missing_classes,
    "source_test_directories_preserved": True,
}

with open(
    OUTPUT_DIR / "dataset_summary.json", "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nYOLO dataset creation complete!")
print("Output:", OUTPUT_DIR)
print("Training images:", len(train_records))
print("Validation images:", len(val_records))
print("Training labels:", len(list(
    (OUTPUT_DIR / "labels/train").glob("*.txt")
)))
print("Validation labels:", len(list(
    (OUTPUT_DIR / "labels/val").glob("*.txt")
)))
print("\nYOLO configuration:")
print((OUTPUT_DIR / "data.yaml").read_text())

Preparing train:   0%|          | 0/1200 [00:00<?, ?image/s]

Preparing val:   0%|          | 0/300 [00:00<?, ?image/s]

Image copying and label generation complete.
Saving audit reports...


Class distribution:   0%|          | 0/21 [00:00<?, ?it/s]


YOLO dataset creation complete!
Output: /kaggle/working/dhakaai_yolo_1500_seed42
Training images: 1200
Validation images: 300
Training labels: 1200
Validation labels: 300

YOLO configuration:
path: "/kaggle/working/dhakaai_yolo_1500_seed42"
train: images/train
val: images/val
nc: 21
names: ["ambulance", "army vehicle", "auto rickshaw", "bicycle", "bus", "car", "garbagevan", "human hauler", "minibus", "minivan", "motorbike", "pickup", "policecar", "rickshaw", "scooter", "suv", "taxi", "three wheelers (CNG)", "truck", "van", "wheelbarrow"]



## Let's validate the generated YOLO dataset

Before training, lets run this final check. It confirms the expected image/label counts, validates YOLO label ranges, and checks that the class IDs are valid.

In [15]:
# Final integrity checks

errors = []

for split in ("train", "val"):
    image_dir = OUTPUT_DIR / "images" / split
    label_dir = OUTPUT_DIR / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ]
    labels = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    if image_stems != label_stems:
        errors.append(
            f"{split}: image/label filename mismatch; "
            f"missing labels={len(image_stems-label_stems)}, "
            f"orphan labels={len(label_stems-image_stems)}"
        )

    for label_file in labels:
        for line_number, line in enumerate(
            label_file.read_text(encoding="utf-8").splitlines(), start=1
        ):
            parts = line.split()
            if len(parts) != 5:
                errors.append(
                    f"{label_file.name}:{line_number}: expected 5 values"
                )
                continue

            try:
                class_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:])

                if not 0 <= class_id < len(class_names):
                    errors.append(
                        f"{label_file.name}:{line_number}: invalid class ID"
                    )

                if not (
                    0 <= xc <= 1 and 0 <= yc <= 1
                    and 0 < bw <= 1 and 0 < bh <= 1
                    and xc - bw / 2 >= -1e-5
                    and yc - bh / 2 >= -1e-5
                    and xc + bw / 2 <= 1 + 1e-5
                    and yc + bh / 2 <= 1 + 1e-5
                ):
                    errors.append(
                        f"{label_file.name}:{line_number}: "
                        "invalid normalized box"
                    )
            except ValueError:
                errors.append(
                    f"{label_file.name}:{line_number}: non-numeric label"
                )

    print(
        f"{split}: {len(images)} images, {len(labels)} labels"
    )

assert len(train_records) == 1200
assert len(val_records) == 300

if errors:
    print("\nValidation errors:", len(errors))
    print(*errors[:20], sep="\n")
    raise ValueError("Dataset validation failed. Fix errors before training.")

print("\nAll integrity checks passed.")
print("Ready for YOLO training:", OUTPUT_DIR / "data.yaml")

train: 1200 images, 1200 labels
val: 300 images, 300 labels

All integrity checks passed.
Ready for YOLO training: /kaggle/working/dhakaai_yolo_1500_seed42/data.yaml


## Review the class distribution
Let's identify classes with limited training or validation coverage.

In [17]:
# Load the saved class-distribution report
distribution_path = OUTPUT_DIR / "class_distribution.csv"

if not distribution_path.exists():
    raise FileNotFoundError(
        f"Class distribution report not found: {distribution_path}"
    )

distribution = pd.read_csv(distribution_path)

# Confirm expected columns exist
required_columns = {"train_objects"}
missing_columns = required_columns - set(distribution.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns in report: {missing_columns}. "
        f"Available columns: {distribution.columns.tolist()}"
    )

# Sort and display the distribution
print("Sorting class distribution...")
distribution = distribution.sort_values(
    "train_objects",
    ascending=True
).reset_index(drop=True)

# Show progress while preparing the rows for display
rows = []

for _, row in tqdm(
    distribution.iterrows(),
    total=len(distribution),
    desc="Preparing class summary",
    unit="class"
):
    rows.append(row.to_dict())

summary = pd.DataFrame(rows)

ipy_display(summary)

print(f"\nTotal classes: {len(summary)}")
print(f"Total training objects: {summary['train_objects'].sum():,}")

if "val_objects" in summary.columns:
    print(f"Total validation objects: {summary['val_objects'].sum():,}")

Sorting class distribution...


Preparing class summary:   0%|          | 0/21 [00:00<?, ?class/s]

,class,train_objects,train_images,val_objects,val_images
0,garbagevan,2,2,1,1
1,army vehicle,13,10,30,29
2,policecar,18,18,14,12
3,scooter,24,23,14,12
4,ambulance,35,35,35,34
5,taxi,40,35,20,17
6,minibus,57,40,38,18
7,wheelbarrow,89,70,30,22
8,human hauler,112,88,57,40
9,auto rickshaw,214,112,158,37



Total classes: 21
Total training objects: 11,101
Total validation objects: 7,012


## Check the split and class coverage

Let's check each class and identifies any classes absent from either split.

In [19]:
distribution = pd.read_csv(OUTPUT_DIR / "class_distribution.csv")

required = {
    "class",
    "train_objects",
    "train_images",
    "val_objects",
    "val_images",
}

missing = required - set(distribution.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

checks = []

for _, row in tqdm(
    distribution.iterrows(),
    total=len(distribution),
    desc="Checking class coverage",
    unit="class"
):
    train_objects = int(row["train_objects"])
    val_objects = int(row["val_objects"])

    checks.append({
        "class": row["class"],
        "train_objects": train_objects,
        "val_objects": val_objects,
        "train_present": train_objects > 0,
        "val_present": val_objects > 0,
        "warning": (
            "No training objects" if train_objects == 0
            else "No validation objects" if val_objects == 0
            else "Very few training objects" if train_objects < 20
            else "OK"
        )
    })

coverage = pd.DataFrame(checks)


from IPython.display import display as ipy_display

print("Class coverage check:")
ipy_display(coverage)

print("\nClasses missing from training:")
missing_train = coverage.loc[
    ~coverage["train_present"], "class"
].tolist()
print(missing_train if missing_train else "None")

print("\nClasses missing from validation:")
missing_val = coverage.loc[
    ~coverage["val_present"], "class"
].tolist()
print(missing_val if missing_val else "None")

print("\nClasses with fewer than 20 training objects:")
rare_classes = coverage.loc[
    coverage["train_objects"] < 20,
    ["class", "train_objects", "val_objects"]
]
ipy_display(rare_classes)

Checking class coverage:   0%|          | 0/21 [00:00<?, ?class/s]

Class coverage check:


,class,train_objects,val_objects,train_present,val_present,warning
0,ambulance,35,35,True,True,OK
1,army vehicle,13,30,True,True,Very few training objects
2,auto rickshaw,214,158,True,True,OK
3,bicycle,302,141,True,True,OK
4,bus,1270,734,True,True,OK
5,car,2489,1833,True,True,OK
6,garbagevan,2,1,True,True,Very few training objects
7,human hauler,112,57,True,True,OK
8,minibus,57,38,True,True,OK
9,minivan,489,349,True,True,OK



Classes missing from training:
None

Classes missing from validation:
None

Classes with fewer than 20 training objects:


,class,train_objects,val_objects
1,army vehicle,13,30
6,garbagevan,2,1
12,policecar,18,14


## One final check before training

Let's make sure the class names and IDs in data.yaml match the IDs written in our YOLO label files.

In [20]:
DATA_YAML = OUTPUT_DIR / "data.yaml"

with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)

print("Number of classes:", config["nc"])
print("Number of class names:", len(config["names"]))
print("Class mapping:")

for class_id, class_name in enumerate(config["names"]):
    print(f"{class_id}: {class_name}")

assert config["nc"] == 21
assert len(config["names"]) == 21
assert len(set(config["names"])) == 21

print("\nConfiguration check passed. Ready for YOLO training.")

Number of classes: 21
Number of class names: 21
Class mapping:
0: ambulance
1: army vehicle
2: auto rickshaw
3: bicycle
4: bus
5: car
6: garbagevan
7: human hauler
8: minibus
9: minivan
10: motorbike
11: pickup
12: policecar
13: rickshaw
14: scooter
15: suv
16: taxi
17: three wheelers (CNG)
18: truck
19: van
20: wheelbarrow

Configuration check passed. Ready for YOLO training.


## Train YOLO

Let's apply `Transfer Learning` technique to train a YOLO (`yolo11n.pt`) pre-trained model rather than starting/building from scratch.

In [22]:
# Train baseline YOLO11n
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(OUTPUT_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    seed=SEED,
    workers=2,
    project="/kaggle/working/runs/detect",
    name="dhakaai_yolo11n_1500",
    patience=10,
    save=True,
    plots=True,
)

print("Best checkpoint:")
print("/kaggle/working/runs/detect/dhakaai_yolo11n_1500/weights/best.pt")

CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dhakaai_yolo_1500_seed42/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dhak

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      3.12G      1.433      2.007      1.158        362        640: 100% ━━━━━━━━━━━━ 75/75 4.4it/s 17.2s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.1it/s 4.7s.3s
                   all        300       7012      0.351      0.143       0.11     0.0607

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      3.12G      1.415      1.799      1.141        303        640: 100% ━━━━━━━━━━━━ 75/75 4.5it/s 16.7s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.2it/s 4.6s.3s
                   all        300       7012      0.289      0.155      0.124     0.0684

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      3.12G      1.388      1.705      1.142        296        640: 100% ━━━━━━━━━━━━ 75/75 4.4it/s 17.0s0.2s
                 Class     Images  Instan

## Let's SAVE the dataset used for future training

Verifying the dataset before saving. The cell below checks that the image/label pairs and configuration file are present.

In [23]:
OUTPUT_DIR = Path("/kaggle/working/dhakaai_yolo_1500_seed42")

for split in ["train", "val"]:
    image_dir = OUTPUT_DIR / "images" / split
    label_dir = OUTPUT_DIR / "labels" / split

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    ]
    labels = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Images without label files:", len(image_stems - label_stems))
    print("Labels without images:", len(label_stems - image_stems))

assert (OUTPUT_DIR / "data.yaml").exists(), "data.yaml is missing"

print("\nDataset configuration:")
print((OUTPUT_DIR / "data.yaml").read_text())


TRAIN
Images: 1200
Labels: 1200
Images without label files: 0
Labels without images: 0

VAL
Images: 300
Labels: 300
Images without label files: 0
Labels without images: 0

Dataset configuration:
path: "/kaggle/working/dhakaai_yolo_1500_seed42"
train: images/train
val: images/val
nc: 21
names: ["ambulance", "army vehicle", "auto rickshaw", "bicycle", "bus", "car", "garbagevan", "human hauler", "minibus", "minivan", "motorbike", "pickup", "policecar", "rickshaw", "scooter", "suv", "taxi", "three wheelers (CNG)", "truck", "van", "wheelbarrow"]



## Packaging the dataset into a ZIP file

The cell below packages the prepared dataset without modifying our original dataset or starting training.

In [24]:
OUTPUT_DIR = Path("/kaggle/working/dhakaai_yolo_1500_seed42")
ZIP_PATH = Path("/kaggle/working/dhakaai_yolo_1500_seed42.zip")

# Collect the files to package
files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file()]

with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as zf:
    for file_path in tqdm(files, desc="Packaging dataset", unit="file"):
        # Preserve the dataset's internal folder structure
        arcname = Path(OUTPUT_DIR.name) / file_path.relative_to(OUTPUT_DIR)
        zf.write(file_path, arcname=str(arcname))

print(f"\nZIP created: {ZIP_PATH}")
print(f"Files packaged: {len(files)}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**3):.2f} GB") 

Packaging dataset:   0%|          | 0/3009 [00:00<?, ?file/s]


ZIP created: /kaggle/working/dhakaai_yolo_1500_seed42.zip
Files packaged: 3009
ZIP size: 0.95 GB
